# Model Comparison

Week 2 comparison of ResNet18, ResNet50, and EfficientNet-B0. The goal is to compare model complexity and prediction quality, not just chase the largest model.

In [ ]:
from pathlib import Path
import json
import pandas as pd
import matplotlib.pyplot as plt

ROOT = Path.cwd()
if not (ROOT / 'reports').exists():
    ROOT = ROOT.parent

FIG_DIR = ROOT / 'reports/figures'
metric_files = {
    'ResNet18': FIG_DIR / 'resnet18_metrics.json',
    'ResNet50': FIG_DIR / 'resnet50' / 'resnet50_metrics.json',
    'EfficientNet-B0': FIG_DIR / 'efficientnet_b0' / 'efficientnet_b0_metrics.json',
}

rows = []
for model_name, path in metric_files.items():
    if path.exists():
        metrics = json.loads(path.read_text())
        rows.append({
            'Model': model_name,
            'Loss': metrics.get('mse'),
            'RMSE': metrics.get('rmse'),
            'MAE': metrics.get('mae'),
            'Metrics path': str(path),
        })
    else:
        rows.append({'Model': model_name, 'Loss': None, 'RMSE': None, 'MAE': None, 'Metrics path': str(path)})

comparison = pd.DataFrame(rows)
comparison

In [ ]:
plot_df = comparison.dropna(subset=['RMSE', 'MAE'])
if not plot_df.empty:
    ax = plot_df.set_index('Model')[['RMSE', 'MAE']].plot(kind='bar', figsize=(8, 4))
    ax.set_ylabel('Error')
    ax.set_title('Model Comparison')
    plt.xticks(rotation=0)
    plt.tight_layout()
    plt.savefig(FIG_DIR / 'model_comparison.png', dpi=160)
    plt.show()
else:
    print('Run evaluate.py for each model before plotting comparison results.')

## Commands

```bash
python src/training/train.py --config configs/resnet18.yaml --no_wandb
python src/training/evaluate.py --config configs/resnet18.yaml --checkpoint models/resnet18_baseline_best.pth

python src/training/train.py --config configs/resnet50.yaml --no_wandb
python src/training/evaluate.py --config configs/resnet50.yaml --checkpoint models/resnet50_comparison_best.pth

python src/training/train.py --config configs/efficientnet_b0.yaml --no_wandb
python src/training/evaluate.py --config configs/efficientnet_b0.yaml --checkpoint models/efficientnet_b0_comparison_best.pth
```

Interview framing: ResNet18 is the baseline, ResNet50 tests whether depth improves accuracy, and EfficientNet-B0 tests parameter-efficient modeling.